# 06 – Valutazione della Pipeline Classica

**Obiettivo**: valutare la qualità e la robustezza dei risultati di anomaly detection  
prima di confrontarli con il pipeline Multi-Agent.

| Sezione | Cosa misuriamo |
|---------|---------------|
| 1. Metriche interne | Silhouette, Davies-Bouldin, Calinski-Harabasz |
| 2. Stabilità | Bootstrap: le stesse rotte rimangono anomale? |
| 3. Sensitivity | Come cambiano i risultati al variare delle soglie? |
| 4. Feature importance | Quali feature guidano i punteggi anomalia? |
| 5. Validazione qualitativa | Le rotte flaggate hanno senso operativo? |
| 6. Scorecard finale | Riepilogo metriche per il confronto Multi-Agent |

**Input**: `anomaly_results.csv` · `features_with_baseline.csv` · `final_report.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

ROOT     = Path('..') 
PROC_DIR = ROOT / 'data' / 'processed'
print('✓ Librerie caricate')

## 1. Caricamento dati

In [ ]:
ar  = pd.read_csv(PROC_DIR / 'anomaly_results.csv')
fb  = pd.read_csv(PROC_DIR / 'features_with_baseline.csv')
rep = pd.read_csv(PROC_DIR / 'final_report.csv')
bs  = json.loads((PROC_DIR / 'baseline_stats.json').read_text())['stats']

ANOMALY_FEATURES = [
    'tot_allarmi_log','pct_interpol','pct_sdi','pct_nsis',
    'tasso_chiusura','tasso_rilevanza','tasso_allarme_medio',
    'tasso_inv_medio','score_rischio_esiti','tasso_respinti','tasso_fermati'
]

X_raw    = fb[ANOMALY_FEATURES].fillna(0).values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Label binaria: anomalia (ALTA+MEDIA) vs normale
labels_ml  = ar['anomaly_label'].map({'ALTA':1,'MEDIA':1,'NORMALE':0}).values
labels_risk = rep['risk_level'].map({'CRITICO':1,'ALTO':1,'MEDIO':1,'BASSO':0}).values

print(f'Feature matrix X : {X_scaled.shape}')
print(f'Anomalie ML (ALTA+MEDIA) : {labels_ml.sum()} ({100*labels_ml.mean():.1f}%)')
print(f'Rotte in allerta (05)    : {labels_risk.sum()} ({100*labels_risk.mean():.1f}%)')

## 2. Metriche Interne di Clustering

Misurano quanto i punti anomali sono *separati* dagli inlier nello spazio delle feature.

- **Silhouette** ∈ [-1, 1]: più alto = cluster ben separati (> 0.3 accettabile, > 0.5 buono)
- **Davies-Bouldin** ≥ 0: più basso = cluster più compatti e distanti (< 1.0 buono)
- **Calinski-Harabasz**: più alto = cluster più densi e separati


In [ ]:
# Silhouette richiede almeno 2 classi con >=2 campioni
# Usiamo le label ML (ALTA+MEDIA=1, NORMALE=0)
sil  = silhouette_score(X_scaled, labels_ml)
db   = davies_bouldin_score(X_scaled, labels_ml)
ch   = calinski_harabasz_score(X_scaled, labels_ml)

# Stesso calcolo con label risk level (05)
sil_r = silhouette_score(X_scaled, labels_risk)
db_r  = davies_bouldin_score(X_scaled, labels_risk)
ch_r  = calinski_harabasz_score(X_scaled, labels_risk)

h1 = "Metrica"
h2 = "Label ML (04)"
h3 = "Label Risk (05)"
print("=== METRICHE INTERNE ===")
print(f"{h1:<25} {h2:<20} {h3}")
print("-" * 65)
rows = [
    ("Silhouette Score",    sil,  sil_r),
    ("Davies-Bouldin Index", db,  db_r),
    ("Calinski-Harabasz",   ch,   ch_r),
]
for name, val_ml, val_r in rows:
    print(f"{name:<25} {val_ml:<20.4f} {val_r:.4f}")
print()
print("Interpretazione Silhouette:")
for val, label in [(sil, "ML"), (sil_r, "Risk")]:
    if   val >= 0.50: interp = "Buona separazione"
    elif val >= 0.25: interp = "Separazione accettabile"
    else:             interp = "Sovrapposizione elevata"
    print(f"  {label}: {val:.4f}  {interp}")


## 3. Analisi di Stabilità (Bootstrap)

Ricampioniamo i dati 100 volte (80% del dataset) e ri-addestriamo IsolationForest.  
Una rotta è **stabile** se appare come anomalia in ≥ 70% delle iterazioni.


In [ ]:
np.random.seed(42)
N_BOOTSTRAP = 100
SAMPLE_FRAC = 0.80
n_samples   = X_scaled.shape[0]
n_sample    = int(n_samples * SAMPLE_FRAC)

# Conta quante volte ogni rotta viene flaggata come anomalia
flag_counts = np.zeros(n_samples, dtype=int)
appeared    = np.zeros(n_samples, dtype=int)  # quante volte è nel campione

for i in range(N_BOOTSTRAP):
    idx = np.random.choice(n_samples, size=n_sample, replace=False)
    Xs  = X_scaled[idx]
    clf = IsolationForest(n_estimators=100, contamination=0.10, random_state=i, n_jobs=-1)
    clf.fit(Xs)
    preds = clf.predict(Xs)  # -1 anomalia, 1 normale
    for j, orig_idx in enumerate(idx):
        appeared[orig_idx] += 1
        if preds[j] == -1:
            flag_counts[orig_idx] += 1

# Frequenza di anomalia per ogni rotta
stability = np.where(appeared > 0, flag_counts / appeared, 0.0)
fb['stability_score'] = stability.round(4)

STABLE_THRESHOLD = 0.70
n_stable = (stability >= STABLE_THRESHOLD).sum()

print(f'Bootstrap completato: {N_BOOTSTRAP} iterazioni')
print(f'Rotte stabili (anomalia in ≥{int(STABLE_THRESHOLD*100)}% campioni): {n_stable}')
print()
print('Top 15 rotte per stabilità:')
stab_df = fb[['ROTTA','PAESE_PART']].copy()
stab_df['stability_score'] = stability
stab_df = stab_df.merge(ar[['ROTTA','anomaly_label','anomaly_score']], on='ROTTA', how='left')
print(stab_df.nlargest(15,'stability_score')[['ROTTA','PAESE_PART','anomaly_label','anomaly_score','stability_score']].to_string(index=False))

In [ ]:
# Overlap: rotte stabili vs rotte anomale secondo ML
ml_anomalies   = set(ar[ar['anomaly_label'].isin(['ALTA','MEDIA'])]['ROTTA'])
stable_routes  = set(stab_df[stab_df['stability_score'] >= STABLE_THRESHOLD]['ROTTA'])
risk_routes    = set(rep[rep['risk_level'].isin(['CRITICO','ALTO','MEDIO'])]['ROTTA'])

overlap_ml_stab  = ml_anomalies & stable_routes
overlap_risk_stab = risk_routes & stable_routes

print('=== OVERLAP STABILITÀ ===')
print(f'Anomalie ML (ALTA+MEDIA)      : {len(ml_anomalies)}')
print(f'Rotte stabili (bootstrap ≥70%): {len(stable_routes)}')
print(f'Rotte in allerta (05)         : {len(risk_routes)}')
print()
print(f'ML ∩ Stable  : {len(overlap_ml_stab)} rotte  {sorted(overlap_ml_stab)}')
print(f'Risk ∩ Stable: {len(overlap_risk_stab)} rotte  {sorted(overlap_risk_stab)[:10]}')
print()
if ml_anomalies:
    precision_stable = len(overlap_ml_stab) / len(ml_anomalies)
    print(f'% anomalie ML confermate da bootstrap: {precision_stable:.1%}')

## 4. Sensitivity Analysis — Soglie

Come cambia la distribuzione ALTA/MEDIA/NORMALE al variare delle soglie di classificazione?


In [ ]:
scores = ar['anomaly_score'].values

# Griglia di soglie
alta_range  = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
media_range = [0.20, 0.25, 0.30, 0.35, 0.40]

header = f"{'ALTA':>8} {'MEDIA':>10} {'#ALTA':>6} {'#MEDIA':>7} {'#NORM':>8} {'Alert%':>8}"
print(header)
print("-" * 52)
for alta_t in alta_range:
    for media_t in media_range:
        if media_t >= alta_t:
            continue
        n_alta  = int((scores >= alta_t).sum())
        n_media = int(((scores >= media_t) & (scores < alta_t)).sum())
        n_norm  = int((scores < media_t).sum())
        alert_pct = 100 * (n_alta + n_media) / len(scores)
        marker = "  <-- scelto" if (abs(alta_t - 0.45) < 0.001 and abs(media_t - 0.30) < 0.001) else ""
        print(f"{alta_t:>8.2f} {media_t:>10.2f} {n_alta:>6} {n_media:>7} {n_norm:>8} {alert_pct:>7.1f}%{marker}")


## 5. Feature Importance

**Permutation importance**: misuriamo quanto peggiora il modello (IsolationForest)  
quando permuto casualmente i valori di una feature. Più degrada → più la feature è importante.


In [ ]:
# Re-fit IsolationForest sul dataset completo
clf_full = IsolationForest(n_estimators=200, contamination=0.10, random_state=42, n_jobs=-1)
clf_full.fit(X_scaled)
base_scores = -clf_full.score_samples(X_scaled)  # più alto = più anomalo

importances = {}
N_PERMS = 30
rng = np.random.RandomState(42)

for j, feat in enumerate(ANOMALY_FEATURES):
    deltas = []
    for _ in range(N_PERMS):
        X_perm = X_scaled.copy()
        X_perm[:, j] = rng.permutation(X_perm[:, j])
        perm_scores = -clf_full.score_samples(X_perm)
        # Quanto cambia il ranking delle anomalie top-10%?
        top_base = set(np.argsort(base_scores)[-57:])   # top 10%
        top_perm = set(np.argsort(perm_scores)[-57:])
        delta = 1.0 - len(top_base & top_perm) / 57
        deltas.append(delta)
    importances[feat] = np.mean(deltas)

imp_df = pd.DataFrame({'feature': list(importances.keys()),
                        'importance': list(importances.values())})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

print('=== FEATURE IMPORTANCE (Permutation — IsolationForest) ===')
print(f'  Metrica: variazione nel top-10% anomalie dopo permutazione')
print()
for _, row in imp_df.iterrows():
    bar = '█' * int(row['importance'] * 40)
    print(f'  {row["feature"]:<25} {row["importance"]:.4f}  {bar}')

## 6. Validazione Qualitativa

Verifichiamo se le rotte classificate come anomale hanno **caratteristiche operative coerenti**  
con un profilo di rischio elevato (alta presenza INTERPOL, esiti negativi, volume allarmi).


In [ ]:
# Confronto statistiche medie: anomalie vs normali
df_eval = fb[ANOMALY_FEATURES + ['ROTTA', 'PAESE_PART']].copy()
df_eval = df_eval.merge(ar[['ROTTA', 'anomaly_label', 'anomaly_score']], on='ROTTA', how='left')
df_eval['is_anomaly'] = df_eval['anomaly_label'].isin(['ALTA', 'MEDIA']).astype(int)

print("=== CONFRONTO ANOMALIE vs NORMALI - Media feature chiave ===")
h = "Feature"
print(f"  {h:<28} {'ANOMALIE':>12} {'NORMALI':>12} {'Ratio':>8}")
print("  " + "-" * 64)
for feat in ANOMALY_FEATURES:
    mean_anom = df_eval[df_eval['is_anomaly'] == 1][feat].mean()
    mean_norm = df_eval[df_eval['is_anomaly'] == 0][feat].mean()
    ratio = mean_anom / mean_norm if mean_norm > 0 else float('inf')
    flag = "  <--" if ratio >= 2.0 else ""
    print(f"  {feat:<28} {mean_anom:>12.4f} {mean_norm:>12.4f} {ratio:>7.2f}x{flag}")
print()
print("  <-- = anomalie con valore >= 2x la media dei normali")


In [ ]:
# Profili rotte ALTA
print('=== PROFILO ROTTE ALTA (top anomalie) ===')
alta_routes = ar[ar['anomaly_label']=='ALTA'].sort_values('anomaly_score', ascending=False)
detail_cols = ['ROTTA','PAESE_PART','anomaly_score'] + ANOMALY_FEATURES[:6]
merged_alta = alta_routes.merge(fb, on='ROTTA', how='left')[detail_cols]
print(merged_alta.to_string(index=False))
print()
print('=== PAESI TOP ANOMALI ===')
paese_anom = df_eval[df_eval['is_anomaly']==1]['PAESE_PART'].value_counts().head(10)
print(paese_anom)

## 7. Scorecard Finale — Pipeline Classica

Riepilogo delle metriche per il confronto con la pipeline Multi-Agent.


In [ ]:
# Calcola precision/recall simulati rispetto alle rotte stabili
n_ml_anomalies = int(labels_ml.sum())
n_stable_total = int((stability >= STABLE_THRESHOLD).sum())
n_overlap      = len(overlap_ml_stab)

# Consistency: % anomalie ML confermate da bootstrap
consistency = n_overlap / n_ml_anomalies if n_ml_anomalies > 0 else 0

# Top-1 confidence
top1_conf = rep['confidence'].max()
top1_route = rep.loc[rep['confidence'].idxmax(), 'ROTTA']

# Alert coverage
n_alert = int((rep['risk_level'].isin(['CRITICO','ALTO','MEDIO'])).sum())
alert_rate = n_alert / len(rep)

scorecard = {
    'pipeline':                  'classical_ml',
    'n_routes_analyzed':         int(len(fb)),
    'n_anomalies_ml':            n_ml_anomalies,
    'n_routes_alert_final':      n_alert,
    'alert_rate_pct':            round(100*alert_rate, 2),
    'silhouette_score':          round(float(sil), 4),
    'davies_bouldin_index':      round(float(db), 4),
    'calinski_harabasz':         round(float(ch), 1),
    'bootstrap_stable_routes':   n_stable_total,
    'ml_bootstrap_consistency':  round(consistency, 4),
    'top1_route':                top1_route,
    'top1_confidence':           round(float(top1_conf), 4),
    'top_feature':               imp_df.iloc[0]['feature'],
    'top_feature_importance':    round(float(imp_df.iloc[0]['importance']), 4),
    'feature_importance_ranking': imp_df[['feature','importance']].to_dict('records'),
}

print('='*55)
print('  SCORECARD — PIPELINE CLASSICA')
print('='*55)
for k, v in scorecard.items():
    if k != 'feature_importance_ranking':
        print(f'  {k:<35} {v}')
print('='*55)

## 8. Export

In [ ]:
import os

# Salva scorecard
scorecard_path = PROC_DIR / 'evaluation_scorecard.json'
with open(scorecard_path, 'w', encoding='utf-8') as f:
    json.dump(scorecard, f, indent=2, ensure_ascii=False)
print(f'✓ evaluation_scorecard.json salvata')

# Salva stability scores per rotta
stab_export = stab_df[['ROTTA','PAESE_PART','anomaly_label','anomaly_score','stability_score']]
stab_export.to_csv(PROC_DIR / 'stability_scores.csv', index=False)
print(f'✓ stability_scores.csv     → {stab_export.shape}')

# Salva feature importance
imp_df.to_csv(PROC_DIR / 'feature_importance.csv', index=False)
print(f'✓ feature_importance.csv   → {imp_df.shape}')

print()
print('Tutti gli output salvati in data/processed/')

## 9. Conclusioni

### Qualità del modello
Le metriche interne (Silhouette, Davies-Bouldin) misurano quanto le anomalie siano
statisticamente distinguibili dalle rotte normali nello spazio delle 11 feature.

### Robustezza
L'analisi di stabilità bootstrap quantifica quante anomalie ML sono confermate
su sottocampioni diversi — una misura diretta della robustezza del segnale.

### Sensibilità
La sensitivity analysis mostra che le soglie ALTA≥0.45 / MEDIA≥0.30 sono conservative
ma giustificate dal range effettivo degli score (max 0.51).

### Feature driving
Le feature più importanti identificano quali dimensioni operative guidano il rischio,
informazione utile per confrontare la spiegabilità con il pipeline Multi-Agent.

---

### Cosa confronteremo con il Multi-Agent

| Dimensione | Classical ML | Multi-Agent LLM |
|------------|-------------|----------------|
| Rotte anomale identificate | *(da questa scorecard)* | *(da costruire)* |
| Alert rate % | *(da questa scorecard)* | *(da costruire)* |
| Overlap rotte flaggate | — | — |
| Spiegabilità | Feature importance | Reasoning testuale |
| Stabilità | Bootstrap score | Consistency multi-run |

---
*Pipeline Classica — Reply × LUISS 2026*